In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,lit,expr,regexp_replace,when,concat,lower
spark = SparkSession.builder.appName("data").getOrCreate()

In [0]:
data = [
    ('101', "Asha",    "South", "Electronics", '1', '45000'),
    ('102', "Ravi",    "North", "Grocery",     '5',  '1200'),
    ('103', "Meena",   "South", "Grocery",     '3',   '900'),
    ('104', "Karthik", "East",  "Electronics", '2', '60000'),
    ('105', "Divya",   "North", "Clothing",    '4',  '3200'),
    ('106', "Imran",   "East",  "Clothing",    '2',  '1800'),
    ('107', "Sneha",   "South", "Electronics", '1', '30000'),
('108', "Arjun",   "North", "Electronics", '1', '52000'),
    ('109', "Kavya",   "East",  "Grocery",     '6',  '1500'),
    ('110', "Rahul",   "South", "Clothing",    '3',  '2700'),
]
cols = ["order_id", "customer", "region", "category", "qty", "amount"]

df =spark.createDataFrame(data,cols)
df.printSchema()



root
 |-- order_id: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- qty: string (nullable = true)
 |-- amount: string (nullable = true)



In [0]:
df = (df.withColumn('order_id',col('order_id').cast('int'))
      .withColumn('qty',col('qty').cast('int'))
      .withColumn('amount',col('amount').cast('int')))

In [0]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- amount: integer (nullable = true)



In [0]:
df.select("customer","category","amount").display()

customer,category,amount
Asha,Electronics,45000
Ravi,Grocery,1200
Meena,Grocery,900
Karthik,Electronics,60000
Divya,Clothing,3200
Imran,Clothing,1800
Sneha,Electronics,30000
Arjun,Electronics,52000
Kavya,Grocery,1500
Rahul,Clothing,2700


In [0]:
df.filter(col("order_id").between(104,108)).display()

order_id,customer,region,category,qty,amount
104,Karthik,East,Electronics,2,60000
105,Divya,North,Clothing,4,3200
106,Imran,East,Clothing,2,1800
107,Sneha,South,Electronics,1,30000
108,Arjun,North,Electronics,1,52000


In [0]:
df.display()

order_id,customer,region,category,qty,amount
101,Asha,South,Electronics,1,45000
102,Ravi,North,Grocery,5,1200
103,Meena,South,Grocery,3,900
104,Karthik,East,Electronics,2,60000
105,Divya,North,Clothing,4,3200
106,Imran,East,Clothing,2,1800
107,Sneha,South,Electronics,1,30000
108,Arjun,North,Electronics,1,52000
109,Kavya,East,Grocery,6,1500
110,Rahul,South,Clothing,3,2700


In [0]:
df.withColumn("region",when(col("order_id").isin(105,108),None)
              .otherwise(col("region")))\
              .withColumn("amount",when(col("order_id").isin(103,107),None)
              .otherwise(col("amount"))).display()
              

order_id,customer,region,category,qty,amount
101,Asha,South,Electronics,1,45000
102,Ravi,North,Grocery,5,1200
103,Meena,South,Grocery,3,null
104,Karthik,East,Electronics,2,60000
105,Divya,null,Clothing,4,3200
106,Imran,East,Clothing,2,1800
107,Sneha,South,Electronics,1,null
108,Arjun,null,Electronics,1,52000
109,Kavya,East,Grocery,6,1500
110,Rahul,South,Clothing,3,2700


In [0]:
df = df.withColumn("level",when(col("amount")>=50000,"High").
              when(col("amount")>=10000,"Medium").
              otherwise("Low"))

In [0]:
df = df.withColumn("Country",lit("India"))

In [0]:
df=df.withColumn("Gmail",lower(concat(col("customer"),lit("123@gmail.com"))))

In [0]:
df.display()

order_id,customer,region,category,qty,amount,level,Country,Gmail
101,Asha,South,Electronics,1,45000,Medium,India,asha123@gmail.com
102,Ravi,North,Grocery,5,1200,Low,India,ravi123@gmail.com
103,Meena,South,Grocery,3,900,Low,India,meena123@gmail.com
104,Karthik,East,Electronics,2,60000,High,India,karthik123@gmail.com
105,Divya,North,Clothing,4,3200,Low,India,divya123@gmail.com
106,Imran,East,Clothing,2,1800,Low,India,imran123@gmail.com
107,Sneha,South,Electronics,1,30000,Medium,India,sneha123@gmail.com
108,Arjun,North,Electronics,1,52000,High,India,arjun123@gmail.com
109,Kavya,East,Grocery,6,1500,Low,India,kavya123@gmail.com
110,Rahul,South,Clothing,3,2700,Low,India,rahul123@gmail.com


In [0]:
df = df.withColumn("Masked_mail",regexp_replace(col("Gmail"),"[123]","*"))



In [0]:
col = [x.upper() for x in df.columns]
df = df.toDF(*col)
df.display()

ORDER_ID,CUSTOMER,REGION,CATEGORY,QTY,AMOUNT,LEVEL,COUNTRY,GMAIL,MASKED_MAIL
101,Asha,South,Electronics,1,45000,Medium,India,asha123@gmail.com,asha***@gmail.com
102,Ravi,North,Grocery,5,1200,Low,India,ravi123@gmail.com,ravi***@gmail.com
103,Meena,South,Grocery,3,900,Low,India,meena123@gmail.com,meena***@gmail.com
104,Karthik,East,Electronics,2,60000,High,India,karthik123@gmail.com,karthik***@gmail.com
105,Divya,North,Clothing,4,3200,Low,India,divya123@gmail.com,divya***@gmail.com
106,Imran,East,Clothing,2,1800,Low,India,imran123@gmail.com,imran***@gmail.com
107,Sneha,South,Electronics,1,30000,Medium,India,sneha123@gmail.com,sneha***@gmail.com
108,Arjun,North,Electronics,1,52000,High,India,arjun123@gmail.com,arjun***@gmail.com
109,Kavya,East,Grocery,6,1500,Low,India,kavya123@gmail.com,kavya***@gmail.com
110,Rahul,South,Clothing,3,2700,Low,India,rahul123@gmail.com,rahul***@gmail.com
